### UDFs
- User Defined Functions (UDF) allow custom logic to be applied row by row in Spark SQL or DataFrames.
    - One row at a time!
    - Each row must be serialized/deserialized

```
from pyspark.sql.functons import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def to_uppercase(text):
    return text.upper() if text else None

df=df.withColumn("upper_name", to_uppercase(df["name"]))

```

- Used when built-in Spark functions aren't sufficient

### UDTF's (User defined table functions)

- One input row may return multiple output rows and columns
- Useful for transforming nested or structured data
    - Expanding JSON, arrays, hierarchical data
- More efficient on large tables

```
from pyspark.sql.functions import udtf
from typing import Iterator, Tuple
import json

@udtf(returnType="user_id STRING, event STRING")
def parse_events(json_str: str) -> Iterator[Tuple[str,str]]:
    data = json.loads(json_str)
    user_id = data.get("user, "unknown")
    for event in data.get("events", []):
        yield (user_id, event)

# Register and use in a Query
df = spark.sql("
SELECT parse_events('\"{\"user\": \"U123\",
\"events\":[\"click\",\"view"]}\" ')")
df.show()


```

- Flatten nested events from JSON event logs (kafka, etc).
- Extracting multiple rows per input row
- Replace explode() with something more efficient for flattening data

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udtf, udf
from pyspark.sql.types import IntegerType
import re

In [3]:
# -----------
# User-Defined Table Function (UDTF)
# -----------

@udtf(returnType = "hashtag: string")
class HashtagExtractor:
    def eval(self, text: str):
        """Extracts hashtags from the input text."""
        if text:
            hashtags = re.findall(r"#\w+", text)
            for hashtag in hashtags:
                yield(hashtag,)

In [12]:
# -----------
# User-Defined Function (UDF)
# -----------

@udf(returnType=IntegerType())
def count_hashtags(text: str):
    """Counts the number of hashtags in the input text. """
    if text:
        return len(re.findall(r"#\w+", text))
    return 0

In [5]:
# Initialize Spark Session
spark = SparkSession.builder \
.appName("Python UDTF and UDF example")\
.config("spark.sql.execution.pythonUDTF.enabled", "true") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-pattern-layout-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/06 15:20:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
#If using in a sparksql setting, we need to register the UDF and UDTF
spark.udtf.register("extract_hashtags", HashtagExtractor)

spark.udf.register("count_hashtags", count_hashtags)


25/05/06 15:35:08 WARN SimpleTableFunctionRegistry: The function extract_hashtags replaced a previously registered function.
25/05/06 15:35:08 WARN SimpleFunctionRegistry: The function count_hashtags replaced a previously registered function.


In [7]:
# Using the UDTF in SQL
print("\nUDTF Example (Extract Hashtags):")
spark.sql("SELECT * FROM extract_hashtags('Welcome to #ApacheSpark and #BigData!')").show()


UDTF Example (Extract Hashtags):


[Stage 0:>                                                          (0 + 1) / 1]

+------------+
|     hashtag|
+------------+
|#ApacheSpark|
|    #BigData|
+------------+



In [14]:
#Using the UDF in SQL
print("\nUDF Example (count hashtags):")
spark.sql("SELECT count_hashtags('Welcome to #ApacheSpark and #BigData!') as hashtag_count").show()


UDF Example (count hashtags):
+-------------+
|hashtag_count|
+-------------+
|            2|
+-------------+



In [16]:
# Using BOTH UDTF and UDF with a DataFrame
data = [("Learning #AI with #ML",), ("Explore #DataScience",), ("No hashtags here",)]
df = spark.createDataFrame(data, ["text"])

In [17]:
df.show()

+--------------------+
|                text|
+--------------------+
|Learning #AI with...|
|Explore #DataScience|
|    No hashtags here|
+--------------------+



In [18]:
# apply UDF in a DataFrame query
df.selectExpr("text", "count_hashtags(text) AS num_hashtags").show()

+--------------------+------------+
|                text|num_hashtags|
+--------------------+------------+
|Learning #AI with...|           2|
|Explore #DataScience|           1|
|    No hashtags here|           0|
+--------------------+------------+



In [19]:
# Apply UDTF with a LATERAL JOIN
print("\n Using UDTF with LATERAL JOIN:")

df.createOrReplaceTempView("tweets")

spark.sql(
    "SELECT text, hashtag FROM tweets, LATERAL extract_hashtags(text)"
).show()


 Using UDTF with LATERAL JOIN:
+--------------------+------------+
|                text|     hashtag|
+--------------------+------------+
|Learning #AI with...|         #AI|
|Learning #AI with...|         #ML|
|Explore #DataScience|#DataScience|
+--------------------+------------+

